# RetinaGuard: Kaggle notebook template

Copy this notebook for each training job (`02_train_quality`, `03_train_grader`, ...).

> RetinaGuard is a research prototype, **not a medical device**.

Settings: GPU T4, internet **on** (needed to install the package), private datasets attached under *Add data*.
Everything written to `/kaggle/working` is kept when the session ends; sessions stop after about 12 h, so save a checkpoint every epoch.

In [ ]:
# --- Configuration -----------------------------------------------------------
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"  # pin a commit SHA for reportable runs
RUN_NAME = "template-run"  # stable name; reusing it resumes the run

In [ ]:
# Install the package. Kaggle images ship torch, so ML extras are not reinstalled here.
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}"

In [ ]:
from pathlib import Path

# Attached datasets are mounted read-only under /kaggle/input/<dataset-slug>.
INPUT_ROOT = Path("/kaggle/input")
for p in sorted(INPUT_ROOT.glob("*")):
    print(p)

In [ ]:
import subprocess

from retinaguard import __version__
from retinaguard.tracking import RunLogger, default_output_dir

run_dir = default_output_dir() / "runs" / RUN_NAME
ckpt_dir = run_dir / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

logger = RunLogger(run_dir)
git_sha = subprocess.run(
    ["git", "ls-remote", REPO_URL, GIT_REF], capture_output=True, text=True
).stdout.split("	")[0]
logger.set_tags({"retinaguard_version": __version__, "git_ref": GIT_REF, "git_sha": git_sha})

In [ ]:
# Resume from the newest checkpoint if this run already has one.
checkpoints = sorted(ckpt_dir.glob("epoch_*.pt"))
start_epoch = int(checkpoints[-1].stem.split("_")[1]) + 1 if checkpoints else 0
print(f"Starting at epoch {start_epoch}")

In [ ]:
# --- Training loop placeholder ----------------------------------------------
# Replace with the real training entry point in Phase 2, e.g.
#   from retinaguard.train.train_grader import main
# Each epoch must save ckpt_dir / f"epoch_{epoch:03d}.pt" and log its metrics.
logger.log_params({"img_size": 512})
for epoch in range(start_epoch, start_epoch + 1):
    logger.log_metrics({"dummy_metric": 0.0}, step=epoch)
print(logger.read_metrics())

After the run, download `/kaggle/working/runs/<RUN_NAME>` (or save the notebook version as a dataset) and import it into the local MLflow server.